# DQL: SELECT, WHERE, GROUP BY, HAVING, ORDER BY

Data Query Language is used to read and shape data. This notebook covers the most common query clauses using the employee and department CSV files.


## Setup

Run this first. It creates a Spark session, finds the CSV files, loads them as DataFrames, and registers SQL temp views.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or beside this notebook.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## SELECT

Use `select` to choose columns. In PySpark you can use the DataFrame API or Spark SQL.


In [ ]:
emp.select("empno", "ename", "job", "sal", "deptno").show(10)

spark.sql("""
SELECT empno, ename, job, sal, deptno
FROM emp
LIMIT 10
""").show()


## WHERE

Use `where` or `filter` to keep only rows that match a condition.


In [ ]:
emp.where(F.col("sal") >= 3000).select("empno", "ename", "job", "sal").show()

spark.sql("""
SELECT empno, ename, job, sal
FROM emp
WHERE sal >= 3000
""").show()


## GROUP BY

Use `groupBy` to collect rows into groups, then calculate summaries for each group.


In [ ]:
emp.groupBy("deptno").agg(
    F.count("*").alias("employee_count"),
    F.round(F.avg("sal"), 2).alias("avg_salary"),
    F.sum("sal").alias("total_salary")
).orderBy("deptno").show()

spark.sql("""
SELECT deptno, COUNT(*) AS employee_count, ROUND(AVG(sal), 2) AS avg_salary, SUM(sal) AS total_salary
FROM emp
GROUP BY deptno
ORDER BY deptno
""").show()


## HAVING

`HAVING` filters grouped results. In the DataFrame API, aggregate first and then filter the summary rows.


In [ ]:
dept_summary = emp.groupBy("deptno").agg(
    F.count("*").alias("employee_count"),
    F.sum("sal").alias("total_salary")
)

dept_summary.where(F.col("total_salary") > 15000).orderBy("total_salary", ascending=False).show()

spark.sql("""
SELECT deptno, COUNT(*) AS employee_count, SUM(sal) AS total_salary
FROM emp
GROUP BY deptno
HAVING SUM(sal) > 15000
ORDER BY total_salary DESC
""").show()


## ORDER BY

Use `orderBy` to sort the final result. Sorting is usually one of the last steps in a query.


In [ ]:
emp.select("empno", "ename", "job", "sal").orderBy(F.col("sal").desc(), F.col("ename")).show(10)

spark.sql("""
SELECT empno, ename, job, sal
FROM emp
ORDER BY sal DESC, ename
LIMIT 10
""").show()
